> **TrustBreast — Notebook 5 of 5.** Table 4. Needs no model files.

# File 5 — FIXED version (leakage on five clinical cohorts, paper Table 4)

## Chalane ka tareeqa
1. Colab → **Runtime → Run all** (GPU zaroori nahi — sirf Random Forest + XGBoost; Drive bhi nahi chahiye)
2. Waqt: takreeban **30–50 minute**
3. **STEP 3 — RESULTS SUMMARY** ka output bhejein.

## Kya hai
- STEP 1 = FIXED leakage code (test sirf asli patients par). Correct protocol mein median imputation, scaling aur SMOTE sirf training fold par fit hote hain (missing values sirf Thyroid mein hain).
- STEP 2 = paper ke numbers: errors removed %, headroom correlation, sign test.
- STEP 3 = purane FIXED run se muqabla (✅/❌) + `table4_leakage.csv` save.

In [ ]:
# Colab: repo clone karo (models/ folder isi mein hai). Local Jupyter par yeh cell kuch nahi karta.
import os
if os.path.exists('/content') and not os.path.isdir('models') and not os.path.isdir('../models'):
    !git clone -q https://github.com/Iqra672-ai/TrustBreast.git /content/TrustBreast
    %cd /content/TrustBreast
    !pip -q install -r requirements.txt


## STEP 1 — Leakage test, 5 cohorts (aapka FIXED code)

In [ ]:
# ============================================================================
# STEP 3b — LEAKAGE KA ASAR KAI DATASETS PAR   ★ SAB SE ZYADA QEEMTI ★
#
# NAYI notebook mein chalao (File 1 ki zaroorat nahi — sab khud download karta hai)
# Waqt: ~30-50 minute
#
# Aapka WBCD wala leakage result NULL tha (+0.70, CI mein zero).
# Wajah shayad yeh ke WBCD sirf halka imbalanced hai (1.68:1).
# Yeh script badhte hue imbalance wale datasets par wahi test dohrata hai.
# Agar asar imbalance ke sath barhta nazar aaye — woh ek ASLI methodological
# nateeja hai, aur ek dataset ke null result se kaheen zyada qeemti.
# ============================================================================
!pip install -q ucimlrepo 2>/dev/null

import numpy as np, warnings
warnings.filterwarnings('ignore')
from sklearn.datasets import fetch_openml, load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy import stats

N_FOLDS, N_REPEATS, SEED = 10, 5, 42

def load_all():
    """Har dataset: (X, y) with y in {0,1}, 1 = minority."""
    out = {}
    d = load_breast_cancer()
    out['WBCD (breast, FNA)'] = (d.data, 1 - d.target)          # 1 = malignant
    for name, oml in [('Pima diabetes', 'diabetes'),
                      ('Haberman survival', 'haberman'),
                      ('Blood transfusion', 'blood-transfusion-service-center'),
                      ('Thyroid (sick)', 'sick')]:
        try:
            f = fetch_openml(oml, version=1, as_frame=True, parser='auto')
            X = f.data.select_dtypes(include=[np.number])
            # NaN yahan rehne do: imputation ab har training fold ke andar hoti hai (run() dekho)
            print(f"  {name:<20} missing values: {int(X.isna().sum().sum())}")
            yv = f.target.astype(str).values
            u, c = np.unique(yv, return_counts=True)
            minority = u[np.argmin(c)]
            out[name] = (np.asarray(X, float), (yv == minority).astype(int))
        except Exception as e:
            print(f"  [skip] {name}: {str(e)[:60]}")
    return out

def run(X, y, leak, seed):
    if leak:
        # LEAKY practice: imputation, scaling aur SMOTE sab split se PEHLE poore cohort par
        Xi = SimpleImputer(strategy='median').fit_transform(X)
        Xs = MinMaxScaler().fit_transform(Xi)
        Xr, yr = SMOTE(random_state=seed).fit_resample(Xs, y)
    else:
        Xr, yr = X, y
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    accs = []
    for tr, va in skf.split(Xr, yr):
        if leak:
            va = va[va < len(y)]      # FIX: sirf asli patients par test (nakli SMOTE rows nahi)
            Xtr, Xva, ytr, yva = Xr[tr], Xr[va], yr[tr], yr[va]
        else:
            # CORRECT: imputer (median) aur scaler sirf training fold par fit
            imp = SimpleImputer(strategy='median').fit(Xr[tr])
            Xtr0, Xva0 = imp.transform(Xr[tr]), imp.transform(Xr[va])
            sc = MinMaxScaler().fit(Xtr0)
            Xtr, Xva = sc.transform(Xtr0), sc.transform(Xva0)
            Xtr, ytr = SMOTE(random_state=seed).fit_resample(Xtr, yr[tr])
            yva = yr[va]
        rf = RandomForestClassifier(n_estimators=300, random_state=seed,
                                    n_jobs=-1).fit(Xtr, ytr)
        xg = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                           random_state=seed, eval_metric='logloss',
                           verbosity=0).fit(Xtr, ytr)
        p = (rf.predict_proba(Xva)[:,1] + xg.predict_proba(Xva)[:,1]) / 2
        accs.append(accuracy_score(yva, (p >= .5).astype(int)))
    return np.array(accs)

print("Datasets load ho rahe...\n")
data = load_all()
rows = []

for name, (X, y) in data.items():
    minority = y.mean()
    ratio = (1 - minority) / minority
    L, C = [], []
    for r in range(N_REPEATS):
        L += list(run(X, y, True,  SEED + r))
        C += list(run(X, y, False, SEED + r))
    L, C = np.array(L)*100, np.array(C)*100
    d = L - C
    nt, ntr = 1/N_FOLDS, 1-1/N_FOLDS
    v = d.var(ddof=1)*(1/len(d) + nt/ntr)
    t = d.mean()/np.sqrt(v) if v > 0 else 0
    pv = 2*(1 - stats.t.cdf(abs(t), df=len(d)-1))
    half = stats.t.ppf(.975, len(d)-1)*np.sqrt(v)
    rows.append((name, len(y), minority*100, ratio, C.mean(), L.mean(),
                 d.mean(), d.mean()-half, d.mean()+half, pv))
    print(f"  ✓ {name:<24} n={len(y):<6} minority {minority*100:.1f}%  "
          f"Δ={d.mean():+.2f}")

print("\n" + "="*100)
print(f"{'Dataset':<24}{'n':>6}{'minority%':>11}{'ratio':>8}"
      f"{'correct':>10}{'leaky':>9}{'Δ':>9}{'95% CI':>20}{'p':>8}")
print("-"*100)
for nm, n, mi, ra, c, l, dd, lo, hi, pv in sorted(rows, key=lambda r: -r[2]):
    print(f"{nm:<24}{n:>6}{mi:>10.1f}%{ra:>7.1f}:1{c:>10.2f}{l:>9.2f}"
          f"{dd:>+9.2f}   [{lo:+.2f}, {hi:+.2f}]{pv:>8.3f}")
print("="*100)
print("Agar Δ imbalance ke sath barhta nazar aaye, to yeh paper ka asal")
print("methodological nateeja hai — ek dataset ke null result se bohot behtar.")


## STEP 2 — Paper ke liye numbers (Section 4.3.1)

In [ ]:
import numpy as np
from scipy import stats
R = sorted(rows, key=lambda r: -r[2])
nm   = [r[0] for r in R]; mino = np.array([r[2] for r in R])
corr = np.array([r[4] for r in R]); d = np.array([r[6] for r in R]); pv = np.array([r[9] for r in R])
head = 100 - corr
print(f"{'cohort':<24}{'headroom':>9}{'Δ':>8}{'p':>8}{'errors removed':>16}")
for a, h, dd, pp in zip(nm, head, d, pv):
    print(f"{a:<24}{h:>9.2f}{dd:>+8.2f}{pp:>8.3f}{dd/h*100:>15.1f}%")
r, pr   = stats.pearsonr(head, d)
rs, ps  = stats.spearmanr(mino, d)
k = int((d > 0).sum())
sign_p  = stats.binomtest(k, len(d), 0.5, alternative='greater').pvalue
print(f"\nSignificant at 0.05: {int((pv < .05).sum())}/{len(pv)}")
print(f"Δ vs headroom: Pearson r = {r:.3f} (p = {pr:.3f}, n = {len(d)})")
print(f"Δ vs minority %: Spearman rho = {rs:.2f} (p = {ps:.3f})")
print(f"Direction: {k}/{len(d)} positive, one-sided sign test p = {sign_p:.3f}")


## STEP 3 — ★ RESULTS SUMMARY + reproducibility check

In [ ]:
import pandas as pd
cols = ['cohort','n','minority_pct','ratio','correct','leaky','delta','ci_lo','ci_hi','p']
t4 = pd.DataFrame(rows, columns=cols).sort_values('minority_pct', ascending=False)
t4.to_csv('table4_leakage.csv', index=False)
before = {'WBCD (breast, FNA)':(96.59,97.17,0.58,0.647), 'Pima diabetes':(75.44,77.56,2.12,0.344),
          'Haberman survival':(67.86,70.73,2.87,0.314), 'Blood transfusion':(71.20,74.86,3.67,0.112),
          'Thyroid (sick)':(98.31,98.55,0.24,0.111)}
print("="*78 + "\nREPRODUCIBILITY CHECK (paper Table 4 se muqabla)\n" + "="*78)
ok_all = len(t4) == 5
for r in t4.itertuples():
    now = (round(r.correct,2), round(r.leaky,2), round(r.delta,2), round(r.p,3))
    b = before.get(r.cohort); ok = b is not None and all(abs(x-y) < 0.006 for x, y in zip(now, b)); ok_all &= ok
    print(f"  {'✅' if ok else '❌'} {r.cohort:<22} now correct {now[0]:.2f} leaky {now[1]:.2f} Δ {now[2]:+.2f} p {now[3]:.3f}"
          f"   before {b}")
print("\nSAB MATCH ✅ — File 5 reproducible hai" if ok_all else "\nKuch badla ❌ — output bhejein")
print("Saved: table4_leakage.csv")
